In [ ]:
# Github/local machine version of this notebook.

# The data is too large to include in a regular git repo - files over 100MB. 
# Data is stored using Git Large File System - GLFS. 
# It should automatically work for anyone working on this repo - if not/there is trouble with commits not working, use "git lfs install".

import os
folder_path = 'data/'
# List files for sanity check
print(os.listdir(folder_path))

['aisles.csv', 'departments.csv', 'orders.csv', 'order_products__prior.csv', 'order_products__train.csv', 'products.csv']


In [4]:
# Begin loading data
import pandas as pd

print("~~~~ Loading... ~~~~")

filepaths = {
    'products': os.path.join(folder_path, 'products.csv'),
    'orders': os.path.join(folder_path, 'orders.csv'),
    'train': os.path.join(folder_path, 'order_products__train.csv'),
    'prior': os.path.join(folder_path, 'order_products__prior.csv'),
    'departments': os.path.join(folder_path, 'departments.csv'),
    'aisles': os.path.join(folder_path, 'aisles.csv')
}

database = {}

print (" Files loaded:")

for dataset, filepath in filepaths.items():
  if os.path.exists(filepath):
    database[dataset] = pd.read_csv(filepath)
    print ("\t", dataset, ":\tloaded successfully!", "\t", os.path.getsize(filepath), "Bytes")
  else:
    print ("\t", dataset, "failed loading.")

print ("~~~~ Load complete. ~~~~")

~~~~ Loading... ~~~~
 Files loaded:
	 products :	loaded successfully! 	 2166953 Bytes
	 orders :	loaded successfully! 	 108968645 Bytes
	 train :	loaded successfully! 	 24680147 Bytes
	 prior :	loaded successfully! 	 577550706 Bytes
	 departments :	loaded successfully! 	 270 Bytes
	 aisles :	loaded successfully! 	 2603 Bytes
~~~~ Load complete. ~~~~


In [5]:
# Printing size information about the data

print ("~~~~Dataframe information: ~~~~")

for dataframe_name, dataframe in database.items():
  print (dataframe_name, "has shape:\t", dataframe.shape)
  print ("\tNull datacells:", dataframe.isnull().sum().sum())
  print ("", dataframe.columns)
  print ("-------------------------------------------------")


~~~~Dataframe information: ~~~~
products has shape:	 (49688, 4)
	Null datacells: 0
 Index(['product_id', 'product_name', 'aisle_id', 'department_id'], dtype='str')
-------------------------------------------------
orders has shape:	 (3421083, 7)
	Null datacells: 206209
 Index(['order_id', 'user_id', 'eval_set', 'order_number', 'order_dow',
       'order_hour_of_day', 'days_since_prior_order'],
      dtype='str')
-------------------------------------------------
train has shape:	 (1384617, 4)
	Null datacells: 0
 Index(['order_id', 'product_id', 'add_to_cart_order', 'reordered'], dtype='str')
-------------------------------------------------
prior has shape:	 (32434489, 4)
	Null datacells: 0
 Index(['order_id', 'product_id', 'add_to_cart_order', 'reordered'], dtype='str')
-------------------------------------------------
departments has shape:	 (21, 2)
	Null datacells: 0
 Index(['department_id', 'department'], dtype='str')
-------------------------------------------------
aisles has shap

Since "orders" has missing values, we need to figure out what to do with them.

Taking a look at the data, most (if not all) the missing values are in the "days since last order" column. We are making the assumption that this means there has been no last order at all; and that row is a customer's first order. Further analyzing of the dataset confirms this: the first order from any customer is missing a days_since_prior_order value.

Thankfully, the solution is simple. While we could remove those rows and lose each user's first order, or fill the cells with an arbitrary or calculated value such as "0" or the average of that user's days since prior order, we do not have to. This is because we have already decided that we will not be using that column in our analysis: so, we can drop it entirely.

We are more focused on the contents of the cart, the user who ordered it, and the order number (IE whether it was the user's first order or second order) rather than the time between orders. We expect this data to give us more useful results: for example, if we find a pattern that people's second order contains more condiments than usual, we may be able to recommend them to people making their second order.

In [6]:
# Dropping columns we do not need.


database['orders'] = database['orders'].drop(columns = ['order_hour_of_day', 'days_since_prior_order', 'order_dow'])
database['train'] = database['train'].drop(columns = ['add_to_cart_order'])
database['prior'] = database['prior'].drop(columns = ['add_to_cart_order'])

print ("~~~~ Data after trims ~~~~")
for dataframe_name, dataframe in database.items():
  print (dataframe_name, "has shape:\t", dataframe.shape)
  print ("\tNull datacells:", dataframe.isnull().sum().sum())
  print ("", dataframe.columns)
  print ("-------------------------------------------------")



~~~~ Data after trims ~~~~
products has shape:	 (49688, 4)
	Null datacells: 0
 Index(['product_id', 'product_name', 'aisle_id', 'department_id'], dtype='str')
-------------------------------------------------
orders has shape:	 (3421083, 4)
	Null datacells: 0
 Index(['order_id', 'user_id', 'eval_set', 'order_number'], dtype='str')
-------------------------------------------------
train has shape:	 (1384617, 3)
	Null datacells: 0
 Index(['order_id', 'product_id', 'reordered'], dtype='str')
-------------------------------------------------
prior has shape:	 (32434489, 3)
	Null datacells: 0
 Index(['order_id', 'product_id', 'reordered'], dtype='str')
-------------------------------------------------
departments has shape:	 (21, 2)
	Null datacells: 0
 Index(['department_id', 'department'], dtype='str')
-------------------------------------------------
aisles has shape:	 (134, 2)
	Null datacells: 0
 Index(['aisle_id', 'aisle'], dtype='str')
-------------------------------------------------


In [7]:
# Process the data
print ("To speed up processing of this large dataset, we can reduce the size of the loaded data.\nThe data is currently int64, but it can be reduced to int32 or even bool for certain columns.")

print ("First, we can check the size of each dataframe in RAM.\n")

for dataframe_name, dataframe in database.items():
  print ("For dataset", dataframe_name)
  print ("Max values are:\n", dataframe.max())
  print (dataframe.info(memory_usage="deep"), "\n")

To speed up processing of this large dataset, we can reduce the size of the loaded data.
The data is currently int64, but it can be reduced to int32 or even bool for certain columns.
First, we can check the size of each dataframe in RAM.

For dataset products
Max values are:
 product_id                                          49688
product_name     with a Splash of Pineapple Coconut Water
aisle_id                                              134
department_id                                          21
dtype: object
<class 'pandas.DataFrame'>
RangeIndex: 49688 entries, 0 to 49687
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   product_id     49688 non-null  int64
 1   product_name   49688 non-null  str  
 2   aisle_id       49688 non-null  int64
 3   department_id  49688 non-null  int64
dtypes: int64(3), str(1)
memory usage: 3.0 MB
None 

For dataset orders
Max values are:
 order_id        3421083
user_id        

Analysis shows the largest dataset is 742.4 MB. While modern computers can handle datasets of this size, in the spirit of learning and the course, we will first reduce the size of the largest dataframes by using smaller datatypes.

Most columns in the data are int64, while their maximum values fit well within an int32. The "reordered" column represents a boolean value, where it is either 0 or 1, which can be turned into a boolean datatype to save even more space. By changing datatypes, we can ideally reduce the size on disk by about half.

The smallest dataframes are only a few kilobytes, up to 31 MB. These are small enough to be easily processed and do not need to be reduced in size.

The datasets we will be optimizing in size are "orders" with a size of 254.4 MB, "prior" with a size of 742.4 MB, and "train" with a size of 31.7 MB - because "train" is simply the final order on record for each user, and we may append it to "prior" later so we have more data for better training.

In [8]:
# Reducing size of the largest dataframes for faster processing.
import numpy as np

database["orders"]["order_id"] = database["orders"]["order_id"].astype(np.int32)
database["orders"]["user_id"] = database["orders"]["user_id"].astype(np.int32)
database["orders"]["order_number"] = database["orders"]["order_number"].astype(np.int32)

print ("For dataset orders, max values are")
print (database["orders"].max())
print (database["orders"].info(memory_usage="deep"), "\n")

database["prior"]["order_id"] = database["prior"]["order_id"].astype(np.int32)
database["prior"]["product_id"] = database["prior"]["product_id"].astype(np.int32)
database["prior"]["reordered"] = database["prior"]["reordered"].astype(bool)

print ("For dataset prior, max values are")
print (database["prior"].max())
print (database["prior"].info(memory_usage="deep"), "\n")

database["train"]["order_id"] = database["train"]["order_id"].astype(np.int32)
database["train"]["product_id"] = database["train"]["product_id"].astype(np.int32)
database["train"]["reordered"] = database["train"]["reordered"].astype(bool)

print ("For dataset train, max values are")
print (database["train"].max())
print (database["train"].info(memory_usage="deep"), "\n")

For dataset orders, max values are
order_id        3421083
user_id          206209
eval_set          train
order_number        100
dtype: object
<class 'pandas.DataFrame'>
RangeIndex: 3421083 entries, 0 to 3421082
Data columns (total 4 columns):
 #   Column        Dtype
---  ------        -----
 0   order_id      int32
 1   user_id       int32
 2   eval_set      str  
 3   order_number  int32
dtypes: int32(3), str(1)
memory usage: 81.5 MB
None 

For dataset prior, max values are
order_id      3421083
product_id      49688
reordered        True
dtype: object
<class 'pandas.DataFrame'>
RangeIndex: 32434489 entries, 0 to 32434488
Data columns (total 3 columns):
 #   Column      Dtype
---  ------      -----
 0   order_id    int32
 1   product_id  int32
 2   reordered   bool 
dtypes: bool(1), int32(2)
memory usage: 278.4 MB
None 

For dataset train, max values are
order_id      3421070
product_id      49688
reordered        True
dtype: object
<class 'pandas.DataFrame'>
RangeIndex: 1384617 e

Now with properly sized data types, the dataframes take much less size on disk. The "orders" dataset went from 254.4 MB to 215.3 MB, a reduction of 15.4%, while the "prior" dataset went from 742.4 MB to 278.4 MB, a 62.5% reduction, and "train" went from 32.7 MB to 11.9 MB, a 63.6% reduction.

This is a good result! The largest dataset now takes about 1/3 of the space it originally did, with absolutely no data loss. The smaller redution for the "orders" dataset is due to its "eval_set" field, which contains text. Since the text is always either "train" or "test", it would be possible to reduce it to a bool; however, for this project, a size of 215 MB is already enough.

Now we will combine "prior" and "train". The "train" dataset is made up of more recent orders from each user, which was pre-split for Kaggle's "Instacart Market Basket Analysis" challenge.

We are working on a project outside of that challenge, so we do not need this data split right now. Having all the data in one dataframe lets us train on more data, which should provide a more accurate model. Using the most data we can is in the spirit of Large-Scale Data Analytics as well.

In [9]:
# Combine datasets

ordered_products = pd.concat([database["prior"], database["train"]], ignore_index=True)


Then we will add the "orders" data to this combined dataset, merging on order_ID to connect each order with its user.

In [10]:
# Combine "orders"

ordered_products = ordered_products.merge(database["orders"], on="order_id", how="inner")

In [11]:
# Finally, we merge with the remaining data so we can see the product names, departments, and aisles.

ordered_products = ordered_products.merge(database["products"], on="product_id", how="left").merge(
    database["aisles"], on="aisle_id", how="left").merge(
    database["departments"], on="department_id", how="left")

display (ordered_products.head())

,order_id,product_id,reordered,user_id,eval_set,order_number,product_name,aisle_id,department_id,aisle,department
0,2,33120,True,202279,prior,3,Organic Egg Whites,86,16,eggs,dairy eggs
1,2,28985,True,202279,prior,3,Michigan Organic Kale,83,4,fresh vegetables,produce
2,2,9327,False,202279,prior,3,Garlic Powder,104,13,spices seasonings,pantry
3,2,45918,True,202279,prior,3,Coconut Butter,19,13,oils vinegars,pantry
4,2,30035,False,202279,prior,3,Natural Sweetener,17,13,baking ingredients,pantry


In [12]:
# Cleanup RAM (only runs once per file load)

try: 
    del database
    print ("Database deleted!")
except:
    print ("Database already deleted!")

Database deleted!


Now that we have the data in a separate object (and have deleted it in the database so we don't have multiple copies of it loaded), we can split it into baskets and transaction histories.

In [13]:
# Split into baskets

baskets = (ordered_products.groupby("order_id", as_index = False)["product_name"].agg(list))
baskets["basket_size"] = baskets["product_name"].apply(len)

display (baskets.head())

,order_id,product_name,basket_size
0,1,"[Bulgarian Yogurt, Organic 4% Milk Fat Whole M...",8
1,2,"[Organic Egg Whites, Michigan Organic Kale, Ga...",9
2,3,[Total 2% with Strawberry Lowfat Greek Straine...,8
3,4,"[Plain Pre-Sliced Bagels, Honey/Lemon Cough Dr...",13
4,5,"[Bag of Organic Bananas, Just Crisp, Parmesan,...",26


In [14]:
# Split into transaction histories (products per user)

transaction_history = (ordered_products.groupby("user_id", as_index = False).agg(
    product_name=("product_name", list), orders_created=("order_id", "nunique"), products_ordered=("product_name", "count")))

display(transaction_history.head())

,user_id,product_name,orders_created,products_ordered
0,1,"[Soda, Original Beef Jerky, Pistachios, Organi...",11,70
1,2,"[Artichoke Spinach Dip, Chipotle Beef & Pork R...",15,226
2,3,"[Part Skim Ricotta Cheese, Organic Baby Spinac...",12,88
3,4,"[Extra-Dry Cava, Original Orange Juice, Vegeta...",5,18
4,5,"[Uncured Genoa Salami, Plain Whole Milk Yogurt...",5,46


With baskets and user transaction histories, we can start grouping users and performing analysis on items frequently bought together.

Next, we will prepare code to trim out infrequent items.

Infrequent items increase complexity of analysis while providing little benefit. Since the items are barely ever ordered, they will be barely ever suggested and add noise that distracts from the items people are actually likely to purchase.

If an item is purchased less than 30 times in over 3.4 million orders, is has a a probability of appearing of less than 1 x 10^-7 %.

In [15]:
# Filter infrequent items.

# In progress.

To avoid having to run all this processing every time we restart the kernel, the processed data will be saved to file. Further steps of the project can directly load the files instead of having to run this again.

In [16]:
# Define a path to save the files to.
# Using parquet files instead of .csv will preserve data typing. We absolutely want this, 
# otherwise our carefully pruned bools and int32s might get turned back into int64s

# Requires pip install pyarrow

import pyarrow as pa
import pyarrow.parquet as pq

save_path = "processed/"

baskets.to_parquet(os.path.join(save_path, "baskets.parquet"))
transaction_history.to_parquet(os.path.join(save_path, "transaction_history.parquet"))